# Cuantización a int8 para TFLite Micro

**Proyecto Final de Ingeniería en Electrónica — FaCENA-UNNE**
(Red de sensores LoRa para detección temprana de incendios)

Toma el modelo que entrena `4.red_entrenada.ipynb` —un `.keras` en float32 que da
P(`normal`), P(`humo`) y P(`incendio`) (softmax)— y produce lo que se compila en el nodo ESP32-S3: un `.tflite`
**full-int8** para `esp-tflite-micro`, el mismo modelo como array de C y un
`cuantizacion.json` con las escalas y la regla de decisión. En el camino mide **al modelo
cuantizado**, que es el que va a correr, y no al float.

Está separado del entrenamiento porque tiene otro ritmo: entrenar lleva decenas de
minutos de GPU; cuantizar, medir y ajustar el umbral lleva algunos minutos de CPU y se
repite cada vez que cambia algo del lado del nodo (la calibración, el umbral, el
objetivo de pérdidas) sin tener que volver a entrenar.

| Entra | Sale (en `salidas/modelos/`) |
|---|---|
| el `cnn_fuego_3clases_*.keras` más reciente | `modelo_int8.tflite`, `modelo_int8.cc`, `cuantizacion.json` |

**Recorrido:** datos y modelo → exportación int8 → evaluación int8 y "como lo ve la
cámara" → umbral de decisión → entregables.

## 1. Datos y modelo

Todo corre en **CPU**: ni la conversión ni el intérprete TFLite usan la GPU, y así este
notebook no depende de la cuDNN ni le disputa la memoria de la GTX 1050 a un
entrenamiento.

Los datos se arman con `modulos/dfire.py`, **el mismo módulo que usa el entrenamiento**:
la calibración int8 y la evaluación del modelo cuantizado tienen que ver las imágenes
preprocesadas exactamente como las vio la red al entrenar (reducción bilineal con
*antialias*, redondeo a uint8). Una diferencia ahí no rompe nada visible: corre los
rangos int8 y la exactitud cae en silencio.

La partición es la oficial del banco, y las métricas que valen son las del subconjunto
**sin gemelos** de entrenamiento, tanto en prueba como en validación (§6bis del
notebook 4). El modelo es el `.keras` más reciente que dejó el notebook 4.

In [ ]:
import os

# Todo en CPU: la conversión y el intérprete TFLite no usan GPU, y así este notebook no
# depende de la cuDNN ni compite por la memoria de la GTX 1050 con un entrenamiento.
# Tiene que ir ANTES de importar TensorFlow.
os.environ["CUDA_VISIBLE_DEVICES"] = ""

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from sklearn.metrics import ConfusionMatrixDisplay, classification_report, confusion_matrix
from tensorflow import keras

MODULOS = Path("../modulos") if Path("../modulos").exists() else Path("modulos")
SALIDAS = MODULOS.parent / "salidas"        # misma raíz que `modulos/`
sys.path.insert(0, str(MODULOS.resolve()))
from dfire import (CATEGORIAS, armar_ds, categoria, indexar, leer_imagen, particion,
                   sin_gemelo)

SEED = 42          # sólo ordena dentro de cada split; la pertenencia es la oficial
BATCH_SIZE = 32
DATA_DIR = MODULOS.parent / "data/datasets/sayedgamal99/smoke-fire-detection-yolo/versions/1/data"
assert DATA_DIR.exists(), f"no está el banco en {DATA_DIR}: correr antes 4.red_entrenada.ipynb"

print("TensorFlow:", tf.__version__, "| dispositivos:", tf.config.list_physical_devices())

In [ ]:
ARCHIVOS, ETIQUETAS, SPLITS = indexar(DATA_DIR)     # ETIQUETAS: (N, 2) [fuego, humo]

# Para cuantizar otra corrida del barrido, fijar RUTA_KERAS a mano.
candidatos_keras = sorted((SALIDAS / "modelos").glob("cnn_fuego_3clases_*.keras"),
                          key=lambda p: p.stat().st_mtime)
assert candidatos_keras, "no hay modelo entrenado: correr antes 4.red_entrenada.ipynb"
RUTA_KERAS = candidatos_keras[-1]
modelo_base = keras.models.load_model(RUTA_KERAS)
# Un `.keras` de otro esquema de salida (p. ej. el multi-etiqueta de 2 salidas)
# cargaría igual y rompería todo lo que sigue sin un error claro.
assert modelo_base.output_shape[-1] == len(CATEGORIAS), (
    f"{RUTA_KERAS.name} tiene {modelo_base.output_shape[-1]} salidas y se esperan "
    f"{len(CATEGORIAS)} {CATEGORIAS}: volver a correr 4.red_entrenada.ipynb")
print(f"modelo: {RUTA_KERAS.name}   salida {modelo_base.output_shape}")

TAM = tuple(modelo_base.input_shape[1:3])  # (alto, ancho): sale del modelo, no se repite a mano

CAT = categoria(ETIQUETAS)                  # (N,): 0 normal, 1 humo, 2 incendio
particiones = particion(SPLITS, SEED)
idx_train, idx_val, idx_test = (particiones[s] for s in ("train", "val", "test"))

# Sin mezclar: las predicciones salen en el orden de idx_val / idx_test.
val_ds = armar_ds(ARCHIVOS[idx_val], CAT[idx_val], BATCH_SIZE, tam=TAM)
test_ds = armar_ds(ARCHIVOS[idx_test], CAT[idx_test], BATCH_SIZE, tam=TAM)
y_true = CAT[idx_test]

# Gemelos de entrenamiento (§6bis del notebook 4): prueba Y validación los tienen, y
# con ellos adentro los números salen optimistas. Train se hashea una sola vez.
LIMPIO = sin_gemelo(ARCHIVOS[idx_train], ARCHIVOS[idx_test])
LIMPIO_VAL = sin_gemelo(ARCHIVOS[idx_train], ARCHIVOS[idx_val])

print(f"train {len(idx_train)}   val {len(idx_val)}   test {len(idx_test)}")
print(f"sin gemelo en entrenamiento: prueba {LIMPIO.sum()} de {len(LIMPIO)}, "
      f"validación {LIMPIO_VAL.sum()} de {len(LIMPIO_VAL)}")

In [ ]:
# Referencia float32: lo que el cuantizado tiene que igualar. Las clases son
# excluyentes (softmax), así que no hay umbral: la predicción es el argmax.
y_prob = modelo_base.predict(test_ds, verbose=0)    # (N, 3): P(normal), P(humo), P(incendio)
y_pred = y_prob.argmax(axis=1)
print(f"{len(y_prob)} imágenes de prueba evaluadas en float32")

## 2. Exportación a TFLite Micro (int8)

El entregable no es el `.keras` sino un `.tflite` **full-int8** que el nodo carga en
`esp-tflite-micro`. La cadena, y por qué cada paso:

1. **`modelo_base.export(..., format="tf_saved_model")`** — `TFLiteConverter.from_keras_model`
   no funciona con Keras 3; el camino válido es pasar por un SavedModel.
2. **`optimizations=[Optimize.DEFAULT]`** — activa la cuantización post-entrenamiento.
3. **`representative_dataset`** — el rango dinámico de cada activación se calibra mirando
   imágenes reales (no se guardan: sólo se observan las activaciones).
4. **`supported_ops=[TFLITE_BUILTINS_INT8]`** — sólo *builtins* int8: ni kernels float ni
   *flex delegate*, que TFLite Micro no tiene.
5. **`inference_input_type/output_type = tf.int8`** — la entrada y la salida del `.tflite`
   ya son int8, así el firmware no convierte nada.

El `.keras` que entra es un *checkpoint* del entrenamiento, no un entregable: al nodo va
el `.tflite`.


In [ ]:
# 500 por categoría: con 100 por clase, el modelo anterior de 3 clases perdía entre
# 3,7 y 5,7 puntos de recall de `humo` al pasar a int8 (ver PTQ, abajo). Más
# calibración es la palanca más barata: no toca el entrenamiento.
N_REPRESENTATIVAS = 500     # por categoría -> 1500 imágenes de TRAIN, nunca de test


def ds_representativo(n_por_clase=N_REPRESENTATIVAS):
    '''Imágenes de entrenamiento sin aumento, 0-255 float32, shape [1, alto, ancho, 3].

    Leídas con `leer_imagen`, la misma función que arma los datos de entrenamiento: si
    la calibración viera otro preprocesamiento, los rangos int8 saldrían de otra
    distribución. Estratificado por categoría: si sólo entraran escenas
    diurnas, las nocturnas saturan en int8 y la exactitud se cae justo de noche.
    '''
    for c in range(len(CATEGORIAS)):
        idx = idx_train[CAT[idx_train] == c][:n_por_clase]
        for ruta in ARCHIVOS[idx]:
            yield [tf.expand_dims(tf.cast(leer_imagen(ruta, TAM), tf.float32), 0)]


print(f"{sum(1 for _ in ds_representativo())} imágenes representativas "
      f"({N_REPRESENTATIVAS} por categoría)")

In [ ]:
import tempfile

# El SavedModel es un paso intermedio del converter, no un entregable: vive lo que
# dura la conversión.
with tempfile.TemporaryDirectory() as tmp:
    modelo_base.export(tmp, format="tf_saved_model")

    conv = tf.lite.TFLiteConverter.from_saved_model(tmp)
    conv.optimizations = [tf.lite.Optimize.DEFAULT]
    conv.representative_dataset = ds_representativo
    conv.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
    conv.inference_input_type = tf.int8
    conv.inference_output_type = tf.int8
    tflite = conv.convert()

print(f"modelo int8: {len(tflite)} bytes ({len(tflite) / 1024:.0f} kB)")


### PTQ *full-int8*, sin QAT

La cuantización es **post-entrenamiento** (`Optimize.DEFAULT`), no *quantization-aware
training*: `tensorflow_model_optimization` todavía no soporta Keras 3. Si la caída de float
a int8 supera ~2 puntos de recall en `incendio` o en `humo` —las dos disparan alerta—,
las palancas son, en orden: más imágenes representativas, α=0.5 (796 kB de pesos,
arena ~1,2 MB en PSRAM: sigue entrando en los 8 MB) o mirar qué capa concentra el error.

El criterio miraba sólo `fuego`, y ahí la caída es chica (~1 punto), así que nunca saltó.
Pero `humo` también dispara alerta y es la que más pierde: con 100 imágenes de
calibración por clase cayó entre 3,7 y 5,7 puntos al pasar a int8 (dos corridas del modelo anterior de 3 clases). Por
eso el dataset representativo pasó a 500 por categoría: es la primera palanca y la más
barata, porque no toca el entrenamiento.

El modo de cuantización es **per-channel**: las convoluciones (incluidas las depthwise)
llevan su propia escala por canal de salida. Eso lo soporta `esp-tflite-micro` con
**ESP-NN**, que es el runtime del nodo. No se usa ESP-DL, que en el S3 cuantiza
per-tensor.


In [ ]:
# Sin delegados: el intérprete de la PC usa XNNPACK por defecto y reportaría un DELEGATE
# que no existe en el firmware. Así se ven las ops reales del archivo.
# La salida softmax agrega SOFTMAX en lugar de LOGISTIC a las ops a registrar.
interprete = tf.lite.Interpreter(
    model_content=tflite,
    experimental_op_resolver_type=(
        tf.lite.experimental.OpResolverType.BUILTIN_WITHOUT_DEFAULT_DELEGATES
    ),
)
ops = sorted({d["op_name"] for d in interprete._get_ops_details()})
print(ops)

# Estas son las ops que hay que registrar en el firmware (ADD y MUL son el `Rescaling`
# plegado a int8). Con esta lista se arma un MicroMutableOpResolver en vez de
# AllOpsResolver: decenas de kB menos de flash y de RAM.


In [ ]:
import json

interprete.allocate_tensors()
det_entrada = interprete.get_input_details()[0]
det_salida = interprete.get_output_details()[0]

cuantizacion = {
    "entrada": {"scale": float(det_entrada["quantization"][0]),
                "zero_point": int(det_entrada["quantization"][1]),
                "shape": [int(d) for d in det_entrada["shape"]]},
    "salida": {"scale": float(det_salida["quantization"][0]),
               "zero_point": int(det_salida["quantization"][1]),
               "shape": [int(d) for d in det_salida["shape"]]},
    "etiquetas": CATEGORIAS,        # softmax: excluyentes, suman 1
    "bytes": len(tflite),
}

ruta_modelos = SALIDAS / "modelos"
ruta_modelos.mkdir(parents=True, exist_ok=True)
(ruta_modelos / "cuantizacion.json").write_text(json.dumps(cuantizacion, indent=2))
print(json.dumps(cuantizacion, indent=2))


In [ ]:
def a_c_array(datos, nombre="modelo_int8"):
    '''Array de C listo para compilar en el firmware.

    `alignas(16)`: TFLM exige alineación a 16 bytes para el buffer del modelo.
    `const`: manda el array a `.rodata`, o sea a flash, y no a RAM.
    '''
    lineas = [f"// Modelo TFLite int8 de {len(datos)} bytes — generado por 5_cuantizacion.ipynb",
              "#include <cstdint>",
              "",
              f"alignas(16) const unsigned char {nombre}[] = {{"]
    for i in range(0, len(datos), 12):
        lineas.append("    " + " ".join(f"0x{b:02x}," for b in datos[i:i + 12]))
    lineas.append("};")
    lineas.append(f"const unsigned int {nombre}_len = {len(datos)};")
    return "\n".join(lineas) + "\n"


(ruta_modelos / "modelo_int8.cc").write_text(a_c_array(tflite))
print(f"{ruta_modelos / 'modelo_int8.cc'}: {len(tflite)} bytes ({len(tflite) / 1024:.0f} kB)")


## 3. Evaluación del modelo int8 (y "como lo ve la cámara")

Al nodo va el `.tflite` int8, no el modelo de Keras: hay que medir **a él**. Se corre el
**mismo** `test_ds` (4306 imágenes, nunca vistas) con `tf.lite.Interpreter`: el mismo
archivo que va a correr el ESP32-S3, ejecutado acá en la PC.

Y una tercera pasada, la que más se parece al campo: D-Fire son fotos web de alta
resolución; el OV2640 entrega QVGA 320×240 en **RGB565** (5 bits de rojo, 6 de verde, 5 de azul).
Cuantizar el color a esa profundidad es el único número que predice el comportamiento real.


In [ ]:
def predecir_int8(imagen):
    '''[P(normal), P(humo), P(incendio)] de una imagen float32 0-255, shape (H, W, 3).'''
    escala, zero_point = det_entrada["quantization"]
    x = np.clip(np.round(imagen / escala + zero_point), -128, 127).astype(np.int8)
    interprete.set_tensor(det_entrada["index"], x[None])
    interprete.invoke()
    escala, zero_point = det_salida["quantization"]
    salida = interprete.get_tensor(det_salida["index"])[0].astype(np.float32)
    return (salida - zero_point) * escala


def evaluar_int8(ds, transformar=None):
    '''Corre el .tflite imagen por imagen sobre todo un dataset.'''
    probabilidades = []
    for lote, _ in ds:
        for imagen in lote.numpy():
            entrada = transformar(imagen) if transformar else imagen
            probabilidades.append(predecir_int8(entrada))
    return np.array(probabilidades)


y_prob_int8 = evaluar_int8(test_ds)
y_pred_int8 = y_prob_int8.argmax(axis=1)
print(f"{len(y_pred_int8)} imágenes evaluadas con el modelo int8")


In [ ]:
for nombre, pred in [("float32", y_pred), ("int8", y_pred_int8)]:
    print(f"=== {nombre} ===")
    print(classification_report(y_true, pred, labels=range(3), target_names=CATEGORIAS,
                                digits=3, zero_division=0))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, (pred, titulo) in zip(axes, [(y_pred, "float32"), (y_pred_int8, "int8")]):
    ConfusionMatrixDisplay(
        confusion_matrix(y_true, pred, labels=range(3),
                         normalize="true"),
        display_labels=CATEGORIAS,
    ).plot(ax=ax, cmap="Blues", colorbar=False, values_format=".2f")
    ax.set_title(f"{titulo} — normalizada por fila")
    ax.tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()

Las tablas de arriba dan recall y precisión **por categoría**, con argmax y sin umbral.
Pero al nodo no le importa la categoría exacta: le importa si manda la alerta.
La regla es **alerta si P(`humo`) + P(`incendio`) ≥ umbral** (§3), y eso separa
dos casos que el recall de `incendio` mete en la misma bolsa: un incendio informado
como humo (la alerta sale igual) y uno visto como normal (la alerta se pierde).
La tabla del umbral de decisión los separa, porque es la que responde la pregunta del
proyecto: **de cada cien incendios, ¿cuántos pasan sin que el nodo diga nada?**

In [ ]:
def a_rgb565(imagen):
    '''Color cuantizado a 5/6/5 bits por canal, como lo entrega el OV2640.

    Las imágenes de `test_ds` ya están a TAM: lo único que falta imitar es la
    profundidad de color del sensor.
    '''
    x = np.clip(np.round(imagen), 0, 255).astype(np.uint16)
    r, g, b = x[..., 0] >> 3, x[..., 1] >> 2, x[..., 2] >> 3      # 5, 6 y 5 bits
    return np.stack([(r << 3) | (r >> 2),                          # 5 bits -> 8
                     (g << 2) | (g >> 4),                          # 6 bits -> 8
                     (b << 3) | (b >> 2)], axis=-1).astype(np.float32)


y_prob_camara = evaluar_int8(test_ds, transformar=a_rgb565)
y_pred_camara = y_prob_camara.argmax(axis=1)
print("=== int8 sobre RGB565, como lo ve la cámara ===")
print(classification_report(y_true, y_pred_camara, labels=range(3),
                            target_names=CATEGORIAS, digits=3, zero_division=0))


In [ ]:
from sklearn.metrics import precision_score, recall_score

variantes = [("float32 (Keras)", y_pred),
             ("int8 (TFLite)", y_pred_int8),
             ("int8 + RGB565 (cámara)", y_pred_camara)]


def tabla(titulo, mascara):
    '''Recall y precisión por categoría (argmax, sin umbral provisorio).'''
    print(f"\n{titulo}   ({np.sum(mascara)} imágenes)")
    print(f"{'variante':<26}"
          + "".join(f"{'recall ' + c:>14}{'precisión ' + c:>17}" for c in CATEGORIAS))
    for nombre, pred in variantes:
        r = recall_score(y_true[mascara], pred[mascara], labels=range(3), average=None,
                         zero_division=0)
        p = precision_score(y_true[mascara], pred[mascara], labels=range(3), average=None,
                            zero_division=0)
        print(f"{nombre:<26}"
              + "".join(f"{r[i]:>14.3f}{p[i]:>17.3f}" for i in range(len(CATEGORIAS))))


tabla("prueba completa", np.ones(len(y_true), bool))
# El número que vale: sin los gemelos del entrenamiento (§6bis del notebook 4) no queda nada que
# memorizar, así que acá se ve la generalización de verdad.
tabla("prueba sin gemelos de entrenamiento", LIMPIO)

print(f"\n.tflite             {len(tflite)} bytes ({len(tflite) / 1024:.0f} kB)")
print("MACs, arena y latencia estimadas: §4 del notebook 4 "
      "(la arena se confirma en placa con arena_used_bytes())")

### Umbral de decisión

Hasta acá la predicción fue el argmax, sin umbral: un valor que nadie eligió. Al nodo le
importa una sola pregunta —¿alerto o no?—, así que la regla es **alerta si
P(`humo`) + P(`incendio`) ≥ umbral**, o sea **P(`normal`) ≤ 1 − umbral**, y el umbral
sale de un objetivo explícito: perder como mucho `OBJETIVO_PERDIDAS` de los eventos
(escenas con humo o con incendio). Entre los umbrales que lo cumplen se toma el más alto,
que es el que menos falsas alarmas dispara. La categoría que se informa por LoRa es el
argmax de las tres salidas.

Cuatro cuidados:

- **Se elige en validación y se mide en prueba.** Elegirlo mirando la prueba es ajustarlo
  a los mismos datos con los que después se lo evalúa.
- **Se elige sobre la validación sin gemelos.** Validación también tiene gemelos de
  entrenamiento (§6bis del notebook 4); con ellos adentro las pérdidas se subestiman y el umbral queda
  corto.
- **Se elige con el modelo desplegado** (int8 + RGB565), no con el float: es su salida la
  que el firmware va a comparar.
- **Viaja al firmware** en `cuantizacion.json`, ya convertido a int8: el nodo compara la
  salida cruda sin decuantizar.

`OBJETIVO_PERDIDAS` es una decisión del proyecto, no de la red: cuántos eventos se tolera
perder a cambio de cuántas falsas alarmas. La tabla muestra las dos caras.

El objetivo se cumple **en validación**, no está garantizado en prueba. Los splits de
D-Fire no tienen exactamente la misma dificultad, y la diferencia entre lo que promete
validación y lo que da prueba es la que hay que esperar al pasar al campo: por eso se
imprimen las dos. Si hace falta margen, la perilla es bajar `OBJETIVO_PERDIDAS`.

In [ ]:
# Regla del nodo: alerta si P(humo) + P(incendio) >= UMBRAL, o sea P(normal) <= 1 - UMBRAL.
# Se elige en VALIDACIÓN sin gemelos y con el modelo desplegado (int8 + RGB565);
# la prueba sólo la mide.
# ponytail: un solo umbral sobre P(normal); si hace falta tratar distinto al humo
# (alertar antes ante incendio), el paso siguiente es un umbral por clase.
OBJETIVO_PERDIDAS = 0.05    # perilla del proyecto: fracción de eventos que se tolera perder

p_val = 1 - evaluar_int8(val_ds, transformar=a_rgb565)[:, 0]   # puntaje de alerta
evento_val = CAT[idx_val] > 0
# Sólo validación sin gemelos (LIMPIO_VAL, §1): con los gemelos adentro las pérdidas
# se subestiman y el umbral queda mal puesto.

candidatos = np.linspace(0, 1, 201)
perdidas_val = np.array([np.mean(p_val[evento_val & LIMPIO_VAL] < u) for u in candidatos])
falsas_val = np.array([np.mean(p_val[~evento_val & LIMPIO_VAL] >= u) for u in candidatos])
# Subir el umbral pierde más eventos y da menos falsas alarmas. Con u = 0 no se pierde
# ninguno, así que siempre hay un candidato válido: se toma el MÁS ALTO que cumple.
cumple = perdidas_val <= OBJETIVO_PERDIDAS
k = np.flatnonzero(cumple)[-1]
UMBRAL = float(candidatos[k])
print(f"validación sin gemelos: umbral {UMBRAL:.3f} -> pierde {perdidas_val[k]:.1%} de los "
      f"eventos, falsa alarma en {falsas_val[k]:.1%} de las escenas sin evento\n")

p_test = 1 - y_prob_camara[:, 0]
incendio = (y_true == 2) & LIMPIO
evento = (y_true > 0) & LIMPIO
sin_evento = (y_true == 0) & LIMPIO
print("prueba sin gemelos, int8 + RGB565:")
print(f"{'regla':<16}{'evento sin alerta':>19}{'incendio sin alerta':>20}"
      f"{'incendio sólo como humo':>24}{'falsa alarma':>14}")
for nombre, u in [("umbral 0,5", 0.5), (f"umbral {UMBRAL:.3f}", UMBRAL)]:
    alerta = p_test >= u
    # Incendio con alerta pero informado como humo: el argmax dijo humo, la alerta sale igual.
    como_humo = alerta & (y_prob_camara.argmax(1) == 1)
    print(f"{nombre:<16}{np.mean(~alerta[evento]):>19.1%}{np.mean(~alerta[incendio]):>20.1%}"
          f"{np.mean(como_humo[incendio]):>24.1%}{np.mean(alerta[sin_evento]):>14.1%}")

# El firmware compara la salida int8 cruda, sin decuantizar. Las tres salidas comparten
# tensor de salida, o sea la misma escala y zero_point:
# p_normal = (q - zp) * escala <= 1 - U  <=>  q <= (1 - U) / escala + zp,
# y como q es entero se trunca: q <= floor((1 - U) / escala + zp).
escala, zero_point = det_salida["quantization"]
cuantizacion["decision"] = {
    "regla": "alerta si salida_int8[0] (normal) <= umbral_int8; categoria = argmax(salida_int8)",
    "umbral": UMBRAL,
    "umbral_int8": int(np.floor((1 - UMBRAL) / escala + zero_point)),
    "objetivo_perdidas_validacion": OBJETIVO_PERDIDAS,
}
(ruta_modelos / "cuantizacion.json").write_text(json.dumps(cuantizacion, indent=2))
print(f"\n{ruta_modelos / 'cuantizacion.json'}: {json.dumps(cuantizacion['decision'])}")

## 4. Entregables

| Archivo | Qué es |
|---|---|
| `salidas/modelos/modelo_int8.cc` | el modelo int8 como array de C (`const`, `alignas(16)`): es lo que se compila en el firmware |
| `salidas/modelos/modelo_int8.tflite` | el mismo modelo, para volver a medirlo sin regenerarlo |
| `salidas/modelos/cuantizacion.json` | escala y `zero_point` de entrada y salida (sin esos dos números el firmware no pasa de int8 a probabilidad) y la **regla de decisión**: alerta si P(`humo`) + P(`incendio`) ≥ umbral (o sea P(`normal`) ≤ 1 − umbral), con el umbral ya convertido a int8 (§3) |

El `.keras` que entró es un *checkpoint* del entrenamiento, no un entregable. Las ops a
registrar en el `MicroMutableOpResolver` salen de §2; el reparto de memoria (flash / SRAM
interna / PSRAM), del §4 del notebook 4.

In [ ]:
(ruta_modelos / "modelo_int8.tflite").write_bytes(tflite)

for ruta in sorted(ruta_modelos.iterdir()):
    print(f"{ruta.name:<45}{ruta.stat().st_size / 1024:>9.1f} kB")

## 5. Conclusiones

- **La métrica de despliegue es la alerta, no el recall por categoría.** Un incendio
  informado como humo no pierde la alerta; uno en el que el nodo no alerta, sí. §3
  separa los dos, y el número que vale es **incendio sin alerta en prueba sin gemelos**,
  medido con el modelo int8 + RGB565.
- **La calibración int8 se estratifica por categoría (500 por categoría, §2).** `humo` fue
  la clase floja y la que más sufrió la cuantización en el modelo anterior de 3 clases:
  al pasar a int8 perdía varias veces lo que perdía `fuego`, mientras que la profundidad
  de color RGB565 les costaba lo mismo a las dos (~1,5 puntos). Hay que confirmar que se
  sostiene con el softmax de 3 clases a QVGA.
- **Despliegue en el nodo: entra en PSRAM.** Con MobileNetV1 α=0.25 a QVGA la arena
  estimada es ~585 kB (va a PSRAM, dentro de los 8 MB) y la latencia estimada ~0,6 s
  (§4 del notebook 4). Son números del modelo analítico, no medidos: falta lo que no se
  puede estimar desde la notebook —latencia y consumo reales en placa,
  `arena_used_bytes()` y el desempeño con la cámara del nodo—.